# 6.1 CD8 analysis - AdditionalPlots
- Writing notes
- Plot 1: Among shared clones, what is the average distribution of cells belonging to the same clone in CSF?
- Plot 2: Additional umaps for the supplemental figures
- Plot 3: PB-biased and CSF-biassed score per cluster
- Plot 4: Large bubble plot
- Plot 5: GZMK quantification ("CSF is dominated by GZMK+ transcriptional states compared to PB")
- Plot 6: Boxplots displaying gene expression for mentioned genes highlighted from the XY log2fc analysis
- Curiosity: How would the CSF-biased score look like if we had used pseudobulk methods per clone?
- Plot 7: "Majority of CSF/PB-biased genes were also higher expressed in CSF and PB, respectively"
- Plot 8: CSF-biased clones over expansion thresholds
- Plot 9: Number of cells in CSF per ml over expansion thresholds

In [ ]:
import anndata as ad
import scanpy as sc
import scanpy.external as sce
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import os
import muon as mu
from scipy.stats import binomtest
from scipy.stats import mannwhitneyu
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from skbio.diversity.alpha import gini_index, shannon, simpson

# Helps to define plotting area
import matplotlib.pyplot as plt
from matplotlib import rcParams 
from matplotlib.pyplot import rc_context
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib import colors
from matplotlib_venn import venn2
from scipy.stats import spearmanr
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from brokenaxes import brokenaxes

#Pseudobulk gene expression
import decoupler as dc
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats

#Statsmodel
from statsmodels.nonparametric.smoothers_lowess import lowess

#Venn
from matplotlib_venn import venn3

#Linear regression
from sklearn.linear_model import LinearRegression
from scipy.stats import spearmanr

#chi-square
from scipy.stats import chi2_contingency

## For making text editable in .pdfs once exported use cairo backend instead of agg
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"
sb.set_theme(style="ticks", font="Arial")

# set seed
np.random.seed(44)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999
plt.rcParams["font.family"] = "Arial"
sb.set_theme(style="ticks", font="Arial")

In [ ]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("fontTools").setLevel(logging.WARNING)

# Loading data

In [ ]:
sdata = mu.read(filename="../../04_data_objects/06_preprocessed_v2/2026_6_CD8_annotated.h5mu")

## How many cells?

In [ ]:
sdata["gex"].obs[["Compartment"]].value_counts()

## Color dictionaries

In [ ]:
cluster_color_dict = {
    # --- TN ---
    "0":  "#dbe9f6",
    "1":  "#a1d99b",
    "2":  "#e2d9f3",
    "11": "#637939",

    # --- TCM ---
    "7":  "#fd8d3c",
    "13": "#74c476",

    # --- TCSF ---
    "18": "#1f4fbf",
    "14": "#f768a1",
    "3":  "#3182bd",
    "5":  "#D7BCA6",
    "16": "#737373",

    # --- TEM ---
    "4":  "#6baed6",
    "8":  "#8b0000",

    # --- TE ---
    "12": "#843c39",
    "6":  "#C00000",

    # --- TEMRA ---
    "9":  "#d6616b",
    "10": "#fde0c5",

    # --- TA ---
    "17": "#39FF14",
    "19": "#54278f",
    "20": "#c7c9ff",
    "22": "#c51b8a",

    # --- NK-like ---
    "21": "#F4E3ED",

    # --- GDT cells ---
    "15": "#fdae6b",
}


## Create a new column with cluster_colors by mappin to leiden_1.8
obs_df = sdata["gex"].obs[["leiden_1.8", "L3_ann"]]
obs_df["cluster_color"] = obs_df["leiden_1.8"].astype(str).map(cluster_color_dict)

## Make another dictionary with L3_ann and cluster_color
obs_df["leiden_1.8"] = obs_df["leiden_1.8"].astype(str)
obs_df["cluster_color"] = obs_df["leiden_1.8"].map(cluster_color_dict)

L3_ann_colors = (
    obs_df.groupby("L3_ann")["cluster_color"]
    .agg(lambda x: x.mode().iloc[0])  # most frequent color
    .to_dict()
)

# Work with MS vs NIC
### Exclude the two IC donors

In [ ]:
sdata_temp = sdata.copy()
mu.pp.filter_obs(sdata_temp["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))

###Clean-up.
#Make sure only the indexes recorded in GEX are used in the o*ther modalities too
index_list = sdata_temp["gex"].obs.index.tolist()

## define a cell index
sdata_temp["airr"].obs["cell_index"] = sdata_temp["airr"].obs.index
sdata_temp["prot"].obs["cell_index"] = sdata_temp["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
sdata_temp["airr"].obs["filter_index"] = pd.Categorical(np.where(sdata_temp["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
sdata_temp["prot"].obs["filter_index"] = pd.Categorical(np.where(sdata_temp["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that sdata_temp["airr"].obs and sdata_temp["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(sdata_temp["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(sdata_temp["prot"], 'filter_index', lambda x: x == "GEX_present")

In [ ]:
sdata_temp["gex"].obs["catpat"]

# Writing notes

In [ ]:
## How many CD8 T cells did I have?
sdata["gex"].obs[["Compartment"]].value_counts()

# Plot 1
- Among shared clones, what is the average distribution of cells belonging to the same clone in CSF?

In [ ]:
## First, define clones in CSF
obs_df = sdata_temp["gex"].obs.copy()

obs_df = obs_df[obs_df["Compartment"]=="CSF"]
obs_df = obs_df[obs_df["clone_handle"].notna()]
csf_list = obs_df["clone_handle"].drop_duplicates().tolist()

## First, define clones in PB
obs_df = sdata_temp["gex"].obs.copy()
obs_df = obs_df[obs_df["Compartment"]=="PB"]
obs_df = obs_df[obs_df["clone_handle"].notna()]
pb_list = obs_df["clone_handle"].drop_duplicates().tolist()

## Take the intersection of the csf and pb lists
shared_list = list(set(csf_list) & set(pb_list))

## Count cells in each Compartment
obs_df = sdata_temp["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].notna()]
obs_df = obs_df[obs_df["clone_handle"].isin(shared_list)]
count_df = obs_df[["clone_handle", "Compartment"]].value_counts().reset_index()
count_df = count_df.pivot(index="clone_handle", columns="Compartment", values="count").reset_index()

## Convert to percentage in CSF
count_df["total"] = count_df["CSF"]+count_df["PB"]
count_df["csf_pct"] = 100* count_df["CSF"] / count_df["total"]

## Group into expansion bins
count_df["expan0"] = np.where(count_df["total"] > 0, True, False)
count_df["expan5"] = np.where(count_df["total"] > 5, True, False)
count_df["expan10"] = np.where(count_df["total"] > 10, True, False)
count_df["expan15"] = np.where(count_df["total"] > 15, True, False)
count_df["expan20"] = np.where(count_df["total"] > 20, True, False)

# Long-format dataframe
plot_df = pd.concat([
    count_df.loc[count_df["expan0"],  ["csf_pct"]].assign(Group="n > 0"),
    count_df.loc[count_df["expan5"],  ["csf_pct"]].assign(Group="n > 5"),
    count_df.loc[count_df["expan10"], ["csf_pct"]].assign(Group="n > 10"),
    count_df.loc[count_df["expan15"], ["csf_pct"]].assign(Group="n > 15"),
    count_df.loc[count_df["expan20"], ["csf_pct"]].assign(Group="n > 20"),
])

# Plot mean ± SD
plt.figure(figsize=(4,3))

sb.barplot(
    data=plot_df,
    x="Group",
    y="csf_pct",
    errorbar="sd",   # standard deviation
    capsize=.3,
    color="lightgray"
)

plt.ylabel("Mean clone-wise % of cells\nin CSFof shared clones")
plt.xlabel("")
plt.ylim(0, 100)
sb.despine()
plt.tight_layout()
plt.savefig("../../05_plots/13_final_plots/XX_Mean_clonewise_pct_in_csf_expansion_thresholds.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
# Add mean labels
means = plot_df.groupby("Group")["csf_pct"].mean()
means

## Extended

In [ ]:
## First, define clones in CSF

obs_df = sdata_temp["gex"].obs.copy()

obs_df = obs_df[obs_df["Compartment"] == "CSF"]
obs_df = obs_df[obs_df["clone_handle"].notna()]

csf_list = obs_df["clone_handle"].drop_duplicates().tolist()


## Define clones in PB

obs_df = sdata_temp["gex"].obs.copy()

obs_df = obs_df[obs_df["Compartment"] == "PB"]
obs_df = obs_df[obs_df["clone_handle"].notna()]

pb_list = obs_df["clone_handle"].drop_duplicates().tolist()


## Intersection of CSF and PB clones

shared_list = list(set(csf_list) & set(pb_list))


## Count cells in each compartment

obs_df = sdata_temp["gex"].obs.copy()

obs_df = obs_df[obs_df["clone_handle"].notna()]
obs_df = obs_df[obs_df["clone_handle"].isin(shared_list)]

count_df = (
    obs_df[["clone_handle", "Compartment"]]
    .value_counts()
    .reset_index()
)

count_df = (
    count_df
    .pivot(
        index="clone_handle",
        columns="Compartment",
        values="count"
    )
    .reset_index()
)


## Add catpat for each clone

clone_catpat = (
    obs_df[["clone_handle", "catpat"]]
    .drop_duplicates()
)

count_df = count_df.merge(
    clone_catpat,
    on="clone_handle",
    how="left"
)


## Convert to percentage in CSF

count_df["total"] = count_df["CSF"] + count_df["PB"]

count_df["csf_pct"] = (
    100 * count_df["CSF"] / count_df["total"]
)


## Group into expansion bins

count_df["expan0"] = count_df["total"] > 0
count_df["expan5"] = count_df["total"] > 5
count_df["expan10"] = count_df["total"] > 10
count_df["expan15"] = count_df["total"] > 15
count_df["expan20"] = count_df["total"] > 20


## Long-format dataframe

plot_df = pd.concat([
    count_df.loc[
        count_df["expan0"],
        ["csf_pct", "catpat"]
    ].assign(Group="n > 0"),

    count_df.loc[
        count_df["expan5"],
        ["csf_pct", "catpat"]
    ].assign(Group="n > 5"),

    count_df.loc[
        count_df["expan10"],
        ["csf_pct", "catpat"]
    ].assign(Group="n > 10"),

    count_df.loc[
        count_df["expan15"],
        ["csf_pct", "catpat"]
    ].assign(Group="n > 15"),

    count_df.loc[
        count_df["expan20"],
        ["csf_pct", "catpat"]
    ].assign(Group="n > 20"),
])


# # Remove all unused categories
for col in plot_df.select_dtypes(include="category").columns:
    plot_df[col] = plot_df[col].cat.remove_unused_categories()


## Set order

plot_df["Group"] = pd.Categorical(
    plot_df["Group"],
    categories=["n > 0", "n > 5", "n > 10", "n > 15", "n > 20"],
    ordered=True
)


## Plot mean ± SD

palette = {
    "NIC": "#FFFFFF",
    "MS": "#000000"
}

fig, ax = plt.subplots(figsize=(5, 3.5))

# Barplot
sb.barplot(
    data=plot_df,
    x="Group",
    y="csf_pct",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    errorbar="sd",
    capsize=.2,
    palette=palette,
    edgecolor="black",
    linewidth=1,
    ax=ax
)

# # Individual clones
# sb.stripplot(
#     data=plot_df,
#     x="Group",
#     y="csf_pct",
#     hue="catpat",
#     hue_order=["NIC", "MS"],
#     dodge=True,
#     palette=palette,
#     size=6,
#     jitter=True,
#     alpha=0.8,
#     edgecolor="black",
#     linewidth=0.5,
#     ax=ax
# )

# Remove duplicate legends
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[:2],
    labels[:2],
    title=None,
    frameon=False
)

ax.set_ylabel("Clone-wise % of cells\nin CSF of shared clones")
ax.set_xlabel("")
ax.set_ylim(0, 100)

sb.despine()
plt.tight_layout()

plt.savefig(
    "../../05_plots/13_final_plots/XX_Mean_clonewise_pct_in_csf_expansion_thresholds.pdf",
    format="pdf",
    dpi=400
)

plt.show()

In [ ]:
## For writing
plot_df.groupby(["catpat", "Group"], observed=True).size().reset_index().sort_values("Group")

In [ ]:
## statistics
# Store results
stats = []

for group in plot_df["Group"].cat.categories:
    
    # Extract MS and NIC values
    ms = plot_df.loc[
        (plot_df["Group"] == group) &
        (plot_df["catpat"] == "MS"),
        "csf_pct"
    ].dropna()

    nic = plot_df.loc[
        (plot_df["Group"] == group) &
        (plot_df["catpat"] == "NIC"),
        "csf_pct"
    ].dropna()

    # Mann-Whitney U test
    U, p = mannwhitneyu(
        ms,
        nic,
        alternative="two-sided"
    )

    stats.append({
        "Group": group,
        "n_MS": len(ms),
        "n_NIC": len(nic),
        "MS_median": ms.median(),
        "NIC_median": nic.median(),
        "U": U,
        "p_value": p
    })


# Convert to dataframe
stats_df = pd.DataFrame(stats)


# FDR correction (Benjamini-Hochberg)
stats_df["p_adj"] = multipletests(
    stats_df["p_value"],
    method="fdr_bh"
)[1]


# Significance
stats_df["significant"] = stats_df["p_adj"] < 0.05

stats_df

# Plot 2: Additional umaps

In [ ]:
## Set export parameters
sc.settings.set_figure_params(figsize=(4.3, 4))

In [ ]:
#Main figure material
fig_markers = sc.pl.umap(sdata["gex"], color=["STAT1", "BACH2", "ZNF683", "PABPC1", "TPT1", "EEF1A1", "PASK", "EEF1B2", "EEF1G"], size = 20, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/11_CD8annotation/30_CD8_ZNF683_additional_umaps.pdf", bbox_inches="tight",  dpi=400)
plt.show()


# Plot 3: PB-biased and CSF-biassed score per cluster

In [ ]:
# --- Export gene scores --- 
CSFbiased_genescore = pd.read_csv("../../06_csv_outputs/DEG/CD8_CSF_biased_filtered_DEGs.csv")
PBbiased_genescore = pd.read_csv("../../06_csv_outputs/DEG/CD8_PB_biased_filtered_DEGs.csv")

print(len(CSFbiased_genescore), " genes were CSF-biased")
print(len(PBbiased_genescore), " genes were CSF-biased")

# --- Calculate scores --- 
sc.tl.score_genes(sdata["gex"], 
                      CSFbiased_genescore["names"].drop_duplicates().tolist(),  
                      score_name='CSF permissive score')

sc.tl.score_genes(sdata["gex"], 
                      PBbiased_genescore["names"].drop_duplicates().tolist(),  
                      score_name='PB resident score')

In [ ]:
# ## get the right compartment 
# sdata_temp = sdata.copy()
# mu.pp.filter_obs(sdata_temp["gex"], "Compartment", lambda x: x == "PB")
# mu.pp.filter_obs(sdata_temp["gex"], "leiden_1.8", lambda x: ~x.isin(["3", "5", "14", "16", "18"])) ## 

# # Remove all unused categories
# for col in sdata_temp["gex"].obs.select_dtypes(include="category").columns:
#     sdata_temp["gex"].obs[col] = sdata_temp["gex"].obs[col].cat.remove_unused_categories()

# Extract obs and sort groups by median
obs = sdata["gex"].obs
medians = obs.groupby("leiden_1.8")["CSF permissive score"].median()
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (6, 2)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="CSF permissive score",
        groupby="leiden_1.8",
        order=sorted_groups,
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,   # no individual dots
        jitter=False,      # optional, just to be clean
        inner="box"        # ⬅️ shows the small boxplot inside the violin
    )

# Rotate x-axis labels
# plt.xticks(rotation=0, ha='right')

# Add horizontal reference line if needed
plt.axhline(y=0, linewidth=0.5, color="black", linestyle="--")
plt.ylabel("CSF-biased score")

# Rasterize violin and box shapes for PDF
# for col in ax.collections:
#     col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/42_leiden_vs_CSFscore.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

In [ ]:
# Extract obs and sort groups by median
obs = sdata["gex"].obs
medians = obs.groupby("L3_ann_overview")["CSF permissive score"].median()
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (4, 1.5)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="CSF permissive score",
        groupby="L3_ann_overview",
        order=sorted_groups,
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,   # no individual dots
        jitter=False,      # optional, just to be clean
        inner="box"        # ⬅️ shows the small boxplot inside the violin
    )

# Rotate x-axis labels
# plt.xticks(rotation=0, ha='right')

# Add horizontal reference line if needed
plt.axhline(y=0, linewidth=0.5, color="black", linestyle="--")
plt.ylabel("CSF-biased score")

# Rasterize violin and box shapes for PDF
# for col in ax.collections:
#     col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/42_L3_vs_CSFscore.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

In [ ]:
# Extract obs and sort groups by median
obs = sdata["gex"].obs
medians = obs.groupby("leiden_1.8")["PB resident score"].median()
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (6, 2)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="PB resident score",
        groupby="leiden_1.8",
        order=sorted_groups,
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,   # no individual dots
        jitter=False,      # optional, just to be clean
        inner="box"        # ⬅️ shows the small boxplot inside the violin
    )

# Rotate x-axis labels
# plt.xticks(rotation=0, ha='right')

# Add horizontal reference line if needed
plt.axhline(y=0, linewidth=0.5, color="black", linestyle="--")
plt.ylabel("PB-biased score")

# Rasterize violin and box shapes for PDF
# for col in ax.collections:
#     col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/42_leiden_vs_PBscore.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

# Plot 4: Large bubble plot

In [ ]:
marker_genes = {
         "CITEseq": ["prot_CD45RA_adt", "prot_CD62L_adt","prot_CD95_adt"],
         "Naive/Memory": ["IL7R", "LTB", "TCF7", "SELL", "CCR7","BACH2", "MRPS6"],
          "Interferon responses": ["ISG15", "MX1", "STAT1", "IFI44"],
         "Co-stimulation" : ["CD27", "CD28", "ITGB2", "ITGAL"],
         "Effector functions": ["CCL5", "NKG7", "GZMB", "PRF1", "GNLY", "GZMA", "GZMK", "GZMH"], 
         "Differentiation": ["CX3CR1", "LAG3", "TIGIT",  "EOMES", "ZEB2", "PDCD1"],
         "HLA" : ["CD74", "HLA-DRB1"], 
         "Innate-receptors" : ["KLRC2", "NCR3"], 
          "Resident memory": ["CXCR6", "XCL1", "ZNF683"],
         "Activation": ["CDKN1B", "CD69", "NR4A1"],
         "Signaling": ["EIF1", "RELA", "TGFB1", "FOS", "JUN",], 
        "Integrins" : ["ITGA4", "ITGB1", "ITGAE", "ITGB7"],
        "Homing" : ["GPR183", "CXCR3", "CXCR4", "CCR4", "CCR5"], 
    "GD T cells": ["TRGV3", "TRGV8","TRDV1"],
    "Ribosomal & mito": ["RPL30", "MT-ATP8"],
}


# # Define a custom order:
cluster_order = [
    "2", "0", "1", "11",
    "7", "13", "21", 
    "4", "8",
   "10",  "9", 
    "6", "12",
    "17", "20", "19","22",
    "15", 
    "18", "14", "3", "5", "16", 
]

# #Implement order
sdata["gex"].obs["leiden_1.8"] = sdata["gex"].obs["leiden_1.8"].cat.reorder_categories(
    cluster_order, ordered=True
)

dp = sc.pl.dotplot(
    sdata["gex"],
    marker_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/11_CD8annotation/5_CD8_bubblechart_supl1.pdf",
    bbox_inches="tight",
    dpi=400
)


# Plot 5: GZMK+ transcriptional states in CSF vs PB

In [ ]:
# Pull expression of GZMK into .obs
sdata["gex"].obs["GZMK_expr"] = sdata["gex"][:, "GZMK"].X.toarray().flatten()

## Count cells and display fraction GZMK+ per donor
obs_df = sdata["gex"].obs[["donor", "catpat", "GZMK_expr", "Compartment"]].copy()

## Per compartment
obs_df["GZMK_cat"] = np.where(obs_df["GZMK_expr"] > 0, "positive", "negative")
plot_df = obs_df[["Compartment", "GZMK_cat", "donor", "catpat"]].value_counts().reset_index().pivot_table(index=["donor", "Compartment", "catpat"], columns="GZMK_cat", values="count", fill_value=0).reset_index()

## % GZMK+ within each compartment
plot_df["pct_gzmk_pos"] = 100*plot_df["positive"] / (plot_df["positive"]+plot_df["negative"])

# Ensure order
plot_df["Compartment"] = pd.Categorical(
    plot_df["Compartment"],
    categories=["PB", "CSF"],
    ordered=True
)

plt.figure(figsize=(3, 4))

# Barplot (mean ± SD)
ax = sb.barplot(
    data=plot_df,
    x="Compartment",
    y="pct_gzmk_pos",
    errorbar="sd",
    capsize=0.15,
    color="lightgrey"
)

# Individual donors
sb.stripplot(
    data=plot_df,
    x="Compartment",
    y="pct_gzmk_pos",
    color="black",
    size=6,
    jitter=False,
    zorder=10,
    ax=ax
)

# Connect paired samples
xpos = {"PB": 0, "CSF": 1}

for donor, df in plot_df.groupby("donor"):
    df = df.sort_values("Compartment")
    ax.plot(
        [xpos[c] for c in df["Compartment"]],
        df["pct_gzmk_pos"],
        color="0.6",
        linewidth=1,
        zorder=1
    )

ax.set_ylabel("% GZMK+ cells")
ax.set_xlabel("")
ax.set_ylim(0, 100)

sb.despine()
plt.tight_layout()
plt.savefig("../../05_plots/13_final_plots/XX_GZMK_expr_in_CSF.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
# Extend compartment
plot_df["Compartment2"]= plot_df["Compartment"].astype(str)+"_"+plot_df["catpat"].astype(str)

plot_df["Compartment2"] = pd.Categorical(
    plot_df["Compartment2"],
    categories=["PB_NIC", "CSF_NIC", "PB_IC", "CSF_IC", "PB_MS", "CSF_MS"],
    ordered=True
)

#Remove IC
plot_df = plot_df[plot_df["catpat"] != "IC"]

# # Remove all unused categories
for col in plot_df.select_dtypes(include="category").columns:
    plot_df[col] = plot_df[col].cat.remove_unused_categories()


plt.figure(figsize=(4, 4))

# Barplot (mean ± SD)
ax = sb.barplot(
    data=plot_df,
    x="Compartment2",
    y="pct_gzmk_pos",
    errorbar="sd",
    capsize=0.15,
    color="lightgrey"
)

# Individual donors
sb.stripplot(
    data=plot_df,
    x="Compartment2",
    y="pct_gzmk_pos",
    color="black",
    size=6,
    jitter=False,
    zorder=10,
    ax=ax
)

# Connect paired samples
xpos = {cat: i for i, cat in enumerate(plot_df["Compartment2"].cat.categories)}

for _, df in plot_df.groupby(["catpat", "donor"]):
    if len(df) == 2:  # PB and CSF both present
        df = df.sort_values("Compartment2")
        ax.plot(
            [xpos[c] for c in df["Compartment2"]],
            df["pct_gzmk_pos"],
            color="0.6",
            linewidth=1,
            zorder=1,
        )
ax.set_ylabel("% GZMK+ cells")
ax.set_xlabel("")
ax.set_ylim(0, 100)

sb.despine()
ax.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/13_final_plots/XX_GZMK+_cells_in_CSF.pdf", format="pdf", dpi=400)
plt.show()

## Multiple genes

In [ ]:
def plot_gene_expression_fraction(
    sdata,
    genes,
    donor_col="donor",
    group_col="catpat",
    compartment_col="Compartment",
    compartments=("PB", "CSF"),
    figsize_per_gene=(3, 4),
    save_path=None
):
    """
    Plot the percentage of cells expressing one or multiple genes per donor
    across paired compartments and perform a paired Wilcoxon signed-rank test.
    """

    # Allow a single gene as input
    if isinstance(genes, str):
        genes = [genes]

    n_genes = len(genes)

    # Create subplots
    fig, axes = plt.subplots(
        1,
        n_genes,
        figsize=(figsize_per_gene[0] * n_genes, figsize_per_gene[1]),
        squeeze=False
    )

    axes = axes.flatten()

    # Get metadata once
    base_obs = sdata["gex"].obs[
        [donor_col, group_col, compartment_col]
    ].copy()

    # Ensure compartment order
    base_obs[compartment_col] = pd.Categorical(
        base_obs[compartment_col],
        categories=list(compartments),
        ordered=True
    )

    # X positions for connecting lines
    xpos = {comp: i for i, comp in enumerate(compartments)}

    for gene, ax in zip(genes, axes):

        # Check that gene exists
        if gene not in sdata["gex"].var_names:
            print(f"Warning: {gene} not found in var_names. Skipping.")
            ax.set_visible(False)
            continue

        # Pull gene expression
        gene_expr = sdata["gex"][:, gene].X

        if hasattr(gene_expr, "toarray"):
            gene_expr = gene_expr.toarray().flatten()
        else:
            gene_expr = np.asarray(gene_expr).flatten()

        # Create temporary dataframe
        obs_df = base_obs.copy()
        obs_df[f"{gene}_expr"] = gene_expr

        # Positive / negative classification
        obs_df[f"{gene}_cat"] = np.where(
            obs_df[f"{gene}_expr"] > 0,
            "positive",
            "negative"
        )

        # Count positive and negative cells per donor/compartment
        plot_df = (
            obs_df[
                [compartment_col, f"{gene}_cat", donor_col, group_col]
            ]
            .value_counts()
            .reset_index(name="count")
            .pivot_table(
                index=[donor_col, compartment_col, group_col],
                columns=f"{gene}_cat",
                values="count",
                fill_value=0
            )
            .reset_index()
        )

        # Make sure both columns exist
        if "positive" not in plot_df.columns:
            plot_df["positive"] = 0

        if "negative" not in plot_df.columns:
            plot_df["negative"] = 0

        # Calculate percentage positive
        plot_df[f"pct_{gene}_pos"] = (
            100
            * plot_df["positive"]
            / (plot_df["positive"] + plot_df["negative"])
        )

        # Barplot: mean ± SD
        sb.barplot(
            data=plot_df,
            x=compartment_col,
            y=f"pct_{gene}_pos",
            order=compartments,
            errorbar="sd",
            capsize=0.15,
            color="lightgrey",
            ax=ax
        )

        # Individual donors
        sb.stripplot(
            data=plot_df,
            x=compartment_col,
            y=f"pct_{gene}_pos",
            order=compartments,
            color="black",
            size=6,
            jitter=False,
            zorder=10,
            ax=ax
        )

        # Connect paired samples
        for donor, donor_df in plot_df.groupby(donor_col):

            donor_df = donor_df.dropna(
                subset=[f"pct_{gene}_pos", compartment_col]
            )

            if len(donor_df) >= 2:
                donor_df = donor_df.sort_values(compartment_col)

                ax.plot(
                    [xpos[c] for c in donor_df[compartment_col]],
                    donor_df[f"pct_{gene}_pos"],
                    color="0.6",
                    linewidth=1,
                    zorder=1
                )

        # ---------------------------------------------------
        # Paired Wilcoxon signed-rank test
        # ---------------------------------------------------

        paired_df = (
            plot_df
            .pivot_table(
                index=donor_col,
                columns=compartment_col,
                values=f"pct_{gene}_pos",
                aggfunc="first"
            )
            .dropna(subset=list(compartments))
        )

        if len(paired_df) >= 2:

            try:
                stat, pval = wilcoxon(
                    paired_df[compartments[0]],
                    paired_df[compartments[1]]
                )

                # Significance label
                if pval < 0.001:
                    p_text = "p < 0.001"
                else:
                    p_text = f"p = {pval:.3f}"

                # Position for annotation
                y_max = max(
                    plot_df[f"pct_{gene}_pos"].max(),
                    1
                )

                y_bracket = min(y_max + 8, 95)
                y_text = min(y_max + 11, 98)

                # Draw bracket
                ax.plot(
                    [0, 0, 1, 1],
                    [
                        y_bracket - 2,
                        y_bracket,
                        y_bracket,
                        y_bracket - 2
                    ],
                    color="black",
                    linewidth=1.2
                )

                # Add p-value
                ax.text(
                    0.5,
                    y_text,
                    p_text,
                    ha="center",
                    va="bottom",
                    fontsize=10
                )

            except ValueError as e:
                print(
                    f"Could not perform Wilcoxon test for {gene}: {e}"
                )

        else:
            print(
                f"Not enough paired donors for Wilcoxon test: {gene}"
            )

        # Formatting
        ax.set_title(gene)
        ax.set_ylabel(f"% {gene}+ cells")
        ax.set_xlabel("")
        ax.set_ylim(0, 100)

        sb.despine(ax=ax)

    plt.tight_layout()

    # Save
    if save_path is not None:
        plt.savefig(
            save_path,
            format="pdf",
            dpi=400,
            bbox_inches="tight"
        )

    plt.show()

    return fig, axes

In [ ]:
genes = ["GZMA", "GZMK", "GZMH", "GZMB", "PRF1", "NKG7", "GNLY", "FASLG"]

fig, axes = plot_gene_expression_fraction(
    sdata,
    genes=genes,
    figsize_per_gene=(3, 4),
    save_path="../../05_plots/13_final_plots/XX_gene_expr_in_CSF.pdf"
)

# Plot 6: Boxplots displaying gene expression for mentioned genes highlighted from the XY log2fc analysis

In [ ]:
# def plot_gene_clone_deg(
#     sdata,
#     gene,
#     min_clone_size=12,
#     donors_exclude=("Pt4", "Pt5"),
#     compartment_colors=None,
#     save_path=None,
# ):
#     """
#     Clone-level DEG-style plot across compartments and patient groups.

#     Parameters
#     ----------
#     sdata : MuData
#     gene : str
#         Gene to analyze.
#     overlap : list
#         Shared clone handles to include.
#     min_clone_size : int
#         Minimum total_clone_count threshold.
#     donors_exclude : tuple
#         Donors to exclude.
#     compartment_colors : dict
#         Color mapping for catpat groups.
#     save_path : str or None
#         Optional output path.
#     """

#     if compartment_colors is None:
#         compartment_colors = {"CSF": "#4B8AC1", "PB": "#C06A6A"}

#     expr_col = f"{gene}_expr"

#     # Copy object
#     sdata_deg = sdata.copy()

#     # Define the shared clones
#     listCSF = sdata_deg["gex"].obs[sdata_deg["gex"].obs["Compartment"] == "CSF"]["clone_handle"].drop_duplicates().tolist()
#     listPB = sdata_deg["gex"].obs[sdata_deg["gex"].obs["Compartment"] == "PB"]["clone_handle"].drop_duplicates().tolist()
    
#     # Clones present in both compartments
#     overlap = list(set(listCSF) & set(listPB))

#     # Filtering
#     ### Includeo only shared clones
#     # mu.pp.filter_obs(
#     #     sdata_deg["gex"],
#     #     "clone_handle",
#     #     lambda x: x.isin(overlap),
#     # )

#     ## Exclude donor pt4 and Pt5
#     mu.pp.filter_obs(
#         sdata_deg["gex"],
#         "donor",
#         lambda x: ~x.isin(list(donors_exclude)),
#     )

#     ### Expansion threshold
#     mu.pp.filter_obs(
#         sdata_deg["gex"],
#         "total_clone_count",
#         lambda x: x > min_clone_size,
#     )
    
#     # Only work with 5 clones per donor to have equal representation (top most expanded)
#     # mu.pp.filter_obs(sdata_deg["gex"], "clone_handle", lambda x: (x.isin(top_clone_handles)))
#     # mu.pp.filter_obs(sdata_deg["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))

#     # Remove unused categories
#     for col in sdata_deg["gex"].obs.select_dtypes(include="category").columns:
#         sdata_deg["gex"].obs[col] = (
#             sdata_deg["gex"].obs[col].cat.remove_unused_categories()
#         )

#     # Expression
#     sdata_deg["gex"].obs[expr_col] = (
#         sdata_deg["gex"][:, gene].X.toarray().flatten()
#     )

#     # Aggregate
#     avg_df = (
#         sdata_deg["gex"].obs[
#             ["donor", "Compartment", "catpat", expr_col]  #Add "clone_handle" if you want per clone 
#         ]
#         .groupby(["donor", "catpat", "Compartment"])[expr_col]  #Add "clone_handle" if you want per clone
#         .mean()
#         .reset_index()
#         .dropna()
#     )

#     num_clones = avg_df["donor"].nunique()   #clone_handle if per clone

#     # Plot
#     g = sb.catplot(
#         data=avg_df,
#         x="catpat",
#         y=expr_col,
#         col="Compartment",
#         kind="box",
#         linewidth=0.5, 
#         color="white", 
#         boxprops=dict(edgecolor="black"),
#         whiskerprops=dict(color="black"),
#         capprops=dict(color="black"),
#         medianprops=dict(color="black"),
#         showfliers=False,
#         height=5,
#         aspect=0.8,
#     )

#     #Remove grid
#     for ax in g.axes.flat:
#         ax.grid(False)

#     # ------------------------
#     # Statistics
#     # ------------------------
    
#     def format_p(p):
#         if np.isnan(p):
#             return "NA"
#         elif p < 0.001:
#             return f"{p:.2e}"
#         else:
#             return f"{p:.3f}"
    
#     stats = {}
    
#     # MS vs NIC within each compartment
#     for compartment in ["CSF", "PB"]:
    
#         sub = avg_df[avg_df["Compartment"] == compartment]
    
#         ms = sub.loc[sub["catpat"] == "MS", expr_col]
#         nic = sub.loc[sub["catpat"] == "NIC", expr_col]
    
#         if len(ms) > 0 and len(nic) > 0:
#             _, p = mannwhitneyu(ms, nic, alternative="two-sided")
#         else:
#             p = np.nan
    
#         stats[compartment] = {"MS_vs_NIC": p}
    
#     # Global CSF vs PB
#     csf = avg_df.loc[avg_df["Compartment"] == "CSF", expr_col]
#     pb = avg_df.loc[avg_df["Compartment"] == "PB", expr_col]
    
#     if len(csf) > 0 and len(pb) > 0:
#         _, p_csf_pb = mannwhitneyu(csf, pb, alternative="two-sided")
#     else:
#         p_csf_pb = np.nan
    
#     stats["CSF_vs_PB"] = p_csf_pb
    
#     # ------------------------
#     # Plot panels
#     # ------------------------
    
#     for ax, compartment in zip(g.axes.flat, g.col_names):
    
#         sub = avg_df[avg_df["Compartment"] == compartment]
    
#         # Connect clones
#         for _, clone in sub.groupby("donor"):  #clone_handle if per clone
#             if clone["catpat"].nunique() == 2:
#                 clone = clone.set_index("catpat").loc[["MS", "NIC"]]
#                 ax.plot(
#                     [0, 1],
#                     clone[expr_col],
#                     color="gray",
#                     alpha=0.6,
#                     lw=1,
#                     zorder=2,
#                 )
    
#         sb.stripplot(
#             data=sub,
#             x="catpat",
#             y=expr_col,
#             color="black",
#             hue="catpat",
#             order=["NIC", "MS"],
#             palette={"NIC": "white", "MS": "black"},
#             edgecolor="black",
#             jitter=0.15,
#             linewidth=0.5,
#             size=5,
#             alpha=0.7,
#             ax=ax,
#             zorder=3,
#         )
    
#         ax.set_title(
#             f"{compartment}\n"
#             f"MS vs NIC: p = {format_p(stats[compartment]['MS_vs_NIC'])}"
#         )
    
#     # Figure title
#     g.fig.suptitle(
#         f"{gene}\n"
#         f"CSF vs PB: p = {format_p(stats['CSF_vs_PB'])}",
#     )

#     #Set figure size
#     g.fig.set_size_inches(4.5, 5)

#     #Set axis labels
#     g.set_axis_labels(" ",f"Avg. expr. / donor")

#     plt.tight_layout()

#     if save_path:
#         plt.savefig(save_path, bbox_inches="tight", dpi=300)

#     plt.show()

#     return avg_df, stats


def plot_gene_clone_deg(
    sdata,
    gene,
    min_clone_size=12,
    donors_exclude=("Pt4", "Pt5"),
    compartment_colors=None,
    save_path=None,
):
    """
    Plot donor-level gene expression across CSF and PB compartments.

    Parameters
    ----------
    sdata : MuData
        MuData object.
    gene : str
        Gene to analyze.
    min_clone_size : int
        Minimum total clone count threshold.
    donors_exclude : tuple
        Donors to exclude.
    compartment_colors : dict
        Color mapping for compartments.
    save_path : str or None
        Optional output path.
    """

    if compartment_colors is None:
        compartment_colors = {"CSF": "#4B8AC1", "PB": "#C06A6A"}

    expr_col = f"{gene}_expr"

    # Copy object
    sdata_deg = sdata.copy()

    # Filter donors
    mu.pp.filter_obs(
        sdata_deg["gex"],
        "donor",
        lambda x: ~x.isin(list(donors_exclude)),
    )

    # Filter expanded clones
    mu.pp.filter_obs(
        sdata_deg["gex"],
        "total_clone_count",
        lambda x: x > min_clone_size,
    )

    # Remove unused categories
    for col in sdata_deg["gex"].obs.select_dtypes(include="category").columns:
        sdata_deg["gex"].obs[col] = (
            sdata_deg["gex"].obs[col].cat.remove_unused_categories()
        )

    # Extract expression
    sdata_deg["gex"].obs[expr_col] = (
        sdata_deg["gex"][:, gene].X.toarray().flatten()
    )

    # Aggregate donor-level expression
    avg_df = (
        sdata_deg["gex"].obs[
            ["donor", "Compartment", expr_col]
        ]
        .groupby(["donor", "Compartment"])[expr_col]
        .mean()
        .reset_index()
        .dropna()
    )

    # Statistics: CSF vs PB
    csf = avg_df.loc[avg_df["Compartment"] == "CSF", expr_col]
    pb = avg_df.loc[avg_df["Compartment"] == "PB", expr_col]

    if len(csf) > 0 and len(pb) > 0:
        _, p_csf_pb = mannwhitneyu(
            csf,
            pb,
            alternative="two-sided"
        )
    else:
        p_csf_pb = np.nan

    def format_p(p):
        if np.isnan(p):
            return "NA"
        elif p < 0.001:
            return f"{p:.2e}"
        else:
            return f"{p:.3f}"

    stats = {"CSF_vs_PB": p_csf_pb}

    # Plot
    plt.figure(figsize=(3, 5))

    ax = sb.boxplot(
        data=avg_df,
        x="Compartment",
        y=expr_col,
        order=["CSF", "PB"],
        palette=compartment_colors,
        linewidth=0.8,
        showfliers=False,
        boxprops=dict(edgecolor="black"),
        whiskerprops=dict(color="black"),
        capprops=dict(color="black"),
        medianprops=dict(color="black"),
    )

    sb.stripplot(
        data=avg_df,
        x="Compartment",
        y=expr_col,
        order=["CSF", "PB"],
        color="black",
        jitter=0.15,
        size=5,
        linewidth=0.5,
        alpha=0.7,
        ax=ax,
    )

    ax.grid(False)

    ax.set_xlabel("")
    ax.set_ylabel("Avg. expr. / donor")

    ax.set_title(
        f"{gene}\n"
        f"CSF vs PB: p = {format_p(p_csf_pb)}"
    )

    plt.tight_layout()

    if save_path:
        plt.savefig(
            save_path,
            bbox_inches="tight",
            dpi=300,
        )

    plt.show()

    return avg_df, stats

#### Examples - PB-biased and CSF-biased genes
""As expected, PB-biased genes were generally more highly expressed in PB than in CSF, whereas CSF-biased genes were enriched in CSF. ""

In [ ]:
genes = ["GZMK", "CD27", "GPR183"]

results = {}

for gene in genes:
    avg_df, stats = plot_gene_clone_deg(
        sdata,
        gene=gene,
        save_path=f"../../05_plots/13_final_plots/{gene}_clone_DEG2.pdf",
    )
    
    results[gene] = {
        "avg_df": avg_df,
        "stats": stats
    }

In [ ]:
genes = ["PPP1R14B", "CREM", "CCR7", "PASK"]

results = {}

for gene in genes:
    avg_df, stats = plot_gene_clone_deg(
        sdata,
        gene=gene,
        save_path=f"../../05_plots/13_final_plots/{gene}_clone_DEG2.pdf",
    )
    
    results[gene] = {
        "avg_df": avg_df,
        "stats": stats
    }

In [ ]:
genes = ["HCST", "S100A4", "MATK", "CTSW", "KIR3DL1"]

results = {}

for gene in genes:
    avg_df, stats = plot_gene_clone_deg(
        sdata,
        gene=gene,
        save_path=f"../../05_plots/13_final_plots/{gene}_clone_DEG2.pdf",
    )
    
    results[gene] = {
        "avg_df": avg_df,
        "stats": stats
    }

In [ ]:
genes = ["GZMB", "GNLY", "GZMH", "PRF1"]

results = {}

for gene in genes:
    avg_df, stats = plot_gene_clone_deg(
        sdata,
        gene=gene,
        save_path=f"../../05_plots/13_final_plots/{gene}_clone_DEG2.pdf",
    )
    
    results[gene] = {
        "avg_df": avg_df,
        "stats": stats
    }

# Curiosity: hyperexpanded T cells in MS vs NIC
https://decoupler.readthedocs.io/en/latest/notebooks/scell/rna_psbk.html

The pseudo-bulk approach involves the following steps:

    Subsetting the cell type of interest

    Extracting their raw integer counts

    Summing their counts per gene into a single profile if they pass quality NIC

Then, DEA can be performed if there are at least two biological replicates per condition (more replicates are recommended).

Pseudobulking can easily be performed using the function decoupler.pp.pseudobulk(). In this example, the counts are just summed, though other modes such as the mean or any custom aggregation function are available. For more information, refer to the mode argument.

In [ ]:
## ---- Define the data ---- 
sdata_bulk = sdata.copy()

mu.pp.filter_obs(sdata_bulk["gex"], "donor", lambda x: ~x.isin(["Pt4", "Pt5"]))
mu.pp.filter_obs(sdata_bulk["gex"], "total_clone_count", lambda x: x > 12)
mu.pp.filter_obs(sdata_bulk["gex"], "Compartment", lambda x: x == "PB")

# Remove all unused categories
for col in sdata_bulk["gex"].obs.select_dtypes(include="category").columns:
    sdata_bulk["gex"].obs[col] = sdata_bulk["gex"].obs[col].cat.remove_unused_categories()

# Create a mask for genes to exclude
genes = sdata_bulk["gex"].var_names
exclude_mask = genes.str.startswith(("TRAV", "TRBV", "TRGV", "TRDV", "TRAJ", "TRBJ", "TRGJ", "TRDJ", "TRAC", "TRBC", "TRGC", "TRDC", "RPS", "RPL", "MT-"))

# Keep only genes not matching the exclusion
genes_to_keep = genes[~exclude_mask]
sdata_bulk = sdata_bulk["gex"][:, genes_to_keep].copy()

## Bring the raw counts into .X
sdata_bulk.X = sdata_bulk.layers["counts"].copy()

In [ ]:
## ---- Use decoupler to pseudobulk cells ----
## --- Define replicates by individual clones --- 
pdata = dc.pp.pseudobulk(
    adata=sdata_bulk,
    sample_col="clone_handle",
    groups_col="catpat",
    mode="sum",
)

pdata

In [ ]:
pdata.obs.head()

In [ ]:
dc.pp.filter_samples(pdata, min_cells=5, min_counts=1000)

In [ ]:
pdata.obs.head()

In [ ]:
# Build DESeq2 object
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    adata=pdata,
    design_factors=["catpat"],
    refit_cooks=True,
    inference=inference,
)

# Compute LFCs
dds.deseq2()

# Extract contrast between conditions
stat_res = DeseqStats(dds, contrast=["catpat", "MS", "NIC"], inference=inference)

# Compute Wald test
stat_res.summary()

In [ ]:
# Extract results
results_df = stat_res.results_df

# filter
results_df = results_df[~results_df["pvalue"].isna()]
results_df = results_df[results_df["baseMean"] > 10]
results_df = results_df[results_df.index != "AC004448.2"] #Exclude a donor-specific transcript, that is strongly expressed in select donors
results_df[results_df["padj"] < 0.05].sort_values("log2FoldChange", ascending=False).head(50)


In [ ]:
dc.pl.volcano(results_df, x="log2FoldChange", y="pvalue", top=20, figsize =(6,6))
plt.savefig("../../05_plots/13_final_plots/49_MSvNIC_PB_hyperexpanded_cells_shared_in_PB_and_CSF.pdf", dpi=400, format="pdf")
plt.show()

# Plot 7: "Majority of CSF/PB-biased genes were also higher expressed in CSF and PB, respectively"

In [ ]:
## From the plotting session for main figure XY graph
plot_df = pd.read_csv("../../06_csv_outputs/DEG/2026_6_CSFvPB_and_CSFbiased_vs_PBbiased_fulltable_non_signif_excluded.csv")

## From the definition of CSFF/PB-biased scores
CSFbiased_genescore = pd.read_csv("../../06_csv_outputs/DEG/CD8_CSF_biased_filtered_DEGs.csv")
PBbiased_genescore = pd.read_csv("../../06_csv_outputs/DEG/CD8_PB_biased_filtered_DEGs.csv")

In [ ]:
### ------- CSF biased genes ----------
## Get a gene liist
CSF_biased_genes = CSFbiased_genescore["names"].tolist()

csf_sub = plot_df[plot_df["gene"].isin(CSF_biased_genes)].copy()

## Annotate CSF-enriched
csf_sub["ann_enr"] = np.where((csf_sub["Compartment_LogFC"] > 0) & (csf_sub["Compartment_p_adj"] < 0.05), "Upregulated in CSF (q < 0.05)", 
                         np.where((csf_sub["Compartment_LogFC"] < 0) & (csf_sub["Compartment_p_adj"] < 0.05), "Upregulated in PB (q < 0.05)", "Non-enriched"))
## number of genes for writing
print(len(csf_sub))
print("CCR7 data:", csf_sub[csf_sub["gene"]=="CCR7"])

## count df
count_csf = csf_sub["ann_enr"].value_counts().reset_index()
count_csf["source"] = "CSF biased score"

### ------- PB biased genes ----------
## Get a gene liist
PB_biased_genes = PBbiased_genescore["names"].tolist()

pb_sub = plot_df[plot_df["gene"].isin(PB_biased_genes)].copy()

## Annotate PB-enriched
pb_sub["ann_enr"] = np.where((pb_sub["Compartment_LogFC"] > 0) & (pb_sub["Compartment_p_adj"] < 0.05), "Upregulated in CSF (q < 0.05)", 
                       np.where((pb_sub["Compartment_LogFC"] < 0) & (pb_sub["Compartment_p_adj"] < 0.05), "Upregulated in PB (q < 0.05)", "Non-enriched"))
## number of genes for writing
print(len(pb_sub))
print("GZMA data:", csf_sub[csf_sub["gene"]=="GZMA"])

## count df
count_pb = pb_sub["ann_enr"].value_counts().reset_index()
count_pb["source"] = "PB biased score"

### ------- Concatenate and plot ----------
plot_df2 = pd.concat([count_pb, count_csf])


# Pivot counts for stacked barplot
plot_df2 = (
    plot_df2
    .pivot(index="source", columns="ann_enr", values="count")
    .fillna(0)
)

## stack order
plot_df2 = plot_df2[["Upregulated in CSF (q < 0.05)", "Non-enriched", "Upregulated in PB (q < 0.05)"]]

# Convert counts to percentages within each source
plot_df2_pct = plot_df2.div(plot_df2.sum(axis=1), axis=0) * 100

# Define colors
colors = {
    "Upregulated in CSF (q < 0.05)": "#4B8AC1",
    "Non-enriched": "lightgrey",
    "Upregulated in PB (q < 0.05)": "#C06A6A",
}

# Plot normalized stacked bars
ax = plot_df2_pct.plot(
    kind="bar",
    stacked=True,
    figsize=(5, 4),
    color=[colors[col] for col in plot_df2_pct.columns],
    edgecolor="black",
    linewidth=0.5,
)
ax.set_ylabel("Percentage of X")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=90)

sb.despine()
plt.grid(False)

plt.legend(
    title="",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.savefig("../../05_plots/13_final_plots/XX_CSF_and_PB_biased_genes_up_in_compartments.pdf")
plt.show()

In [ ]:
## Which examples are outliers? PB
pb_sub.sort_values("Compartment_LogFC", ascending=False).head(5)

In [ ]:
## Which examples are outliers? CSF
csf_sub.sort_values("Compartment_LogFC", ascending=True).head(5)

# Plot 8: CSF-biased scores with flexible thresholds
- Subgoal 1: Find statistical threholds for CSF-biased clones per n_cells belonging to the same clone
- Subgoal 2: Calculate percentage of clones that are CSF-biased
- Subgoal 3: Plot the % of clones that are CSF-biased

#### Subgoal 1: Find statistical threshold for CSF-biased clones

In [ ]:
obs_df = sdata["gex"].obs[["donor", "catpat", "clone_handle", "Compartment"]]

#Filter - exclude Pt4 and Pt5
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]

#calculate the number of cells in CSF for all appropriate donors first (no catpat subset)
compart_sum = obs_df.groupby(["Compartment"]).size().reset_index(name="count")
CSF_cells = compart_sum.loc[compart_sum["Compartment"] == "CSF", "count"].iloc[0]
PB_cells = compart_sum.loc[compart_sum["Compartment"] == "PB", "count"].iloc[0]

#Calculate the fraction of cells in CSF of the data
CSF_pct = CSF_cells / (CSF_cells + PB_cells)
CSF_pct

#For grouped results
compart_sum_grp = obs_df.groupby(["Compartment", "catpat"]).size().reset_index(name="count").pivot(index = "catpat", columns="Compartment", values="count")
compart_sum_grp["total"] = compart_sum_grp["CSF"] + compart_sum_grp["PB"]  
compart_sum_grp["CSF.pct"] = compart_sum_grp["CSF"] / compart_sum_grp["total"] 
compart_sum_grp = compart_sum_grp.reset_index()

In [ ]:
CSF_pct #percentage of the dataset that is CSF cells

In [ ]:
## Given these statistics, make permutations to indicate what is highly significant
# Parameters
p_csf = CSF_pct  # compart_sum.loc[compart_sum["catpat"] == "MS", "CSF.pct"].iloc[0]
alpha = 0.05
max_cells = 80

rows = []

for n in range(1, max_cells + 1):
    for k in range(0, n + 1):
        purity = k / n

        # Binomial test under null
        pval = binomtest(k, n, p_csf, alternative="two-sided").pvalue

        if pval <= alpha:
            rows.append({
                "n_cells": n,
                "purity": purity
            })

sig_df = pd.DataFrame(rows)

In [ ]:
## To get to CSF-biased thresholds take everythin above 0.2
### 
csf_thresholds = sig_df[sig_df["purity"]>0.2].groupby("n_cells").min().reset_index()
csf_thresholds = csf_thresholds.rename(columns={"n_cells" : "total_clone_count", "purity" : "csf_threshold"})
csf_thresholds.head(5)

In [ ]:
## To get to PB-biased thresholds take everythin below 0.2
### 
pb_thresholds = sig_df[sig_df["purity"]<0.2].groupby("n_cells").max().reset_index()
pb_thresholds = pb_thresholds.rename(columns={"n_cells" : "total_clone_count", "purity" : "pb_threshold"})
pb_thresholds.tail(5)

#### Subgoal 2: Calculate percentage of clones that are CSF-biased

In [ ]:
## Get obs_df
obs_df = sdata["gex"].obs
obs_df = obs_df[obs_df["catpat"] != "IC"]
obs_df = obs_df[obs_df["clone_handle"].notna()]

## Select columns
obs_df = obs_df[["donor", "catpat", "clone_handle", "Compartment", "type_x_clone_count", "total_clone_count"]]

# Remove all unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = obs_df[col].cat.remove_unused_categories()

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Compute Compartment Percentages
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Compute % contribution of each compartment per clone
obs_df["Compartment_pct"] = 100 * obs_df["type_x_clone_count"] / obs_df["total_clone_count"]

# Create input dataframe for pivoting
df_input = obs_df[["donor", "clone_handle", "total_clone_count", "Compartment", "Compartment_pct", "catpat"]].drop_duplicates()

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Add flexible statistical thresholds
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
## Add csf_threshold and pb_threshold
df_input = df_input.merge(csf_thresholds).merge(pb_thresholds)

### Problem(fixed): Compartment = PB can be 100% for the Compartment-pct, but what we want is % CSF instead to be able to use the pb_thresholds that are below 0.2 to be significantly pb biased
### Where compartment == PB    modify the compartment pct to be a csf_percentage
df_input["csf_comp_pct"] = np.where(df_input["Compartment"] == "PB", 100-df_input["Compartment_pct"], df_input["Compartment_pct"])

## Prune the columns
df_input = df_input[["donor", "Compartment", "clone_handle", "total_clone_count", "csf_comp_pct", "catpat", "csf_threshold", "pb_threshold"]].drop_duplicates()

## Define lists of CSF-biased and PB-biased clones separately
csf_list = df_input[(df_input["Compartment"] == "CSF") & (df_input["csf_comp_pct"] >= 100*df_input["csf_threshold"])]["clone_handle"].tolist()
pb_list = df_input[(df_input["Compartment"] == "PB") & (df_input["csf_comp_pct"] <= 100*df_input["pb_threshold"])]["clone_handle"].tolist()

## annotate biased_ann
df_input["biased_ann"] = np.where(df_input["clone_handle"].isin(csf_list), "csf_biased", 
                                  np.where(df_input["clone_handle"].isin(pb_list), "pb_biased", "neutral"))

## Prune columns / drop duplicates
df_input = df_input[["donor", "catpat", "clone_handle", "total_clone_count", "csf_comp_pct",  "csf_threshold", "pb_threshold", "biased_ann"]].drop_duplicates()

# # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# # Enumerate per donor how many CSF-biased clones each donor has
# # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
count_df = df_input[["donor", "catpat", "biased_ann"]].value_counts().reset_index()
count_df = count_df.pivot_table(index = ["donor", "catpat"], columns = "biased_ann", values = "count", fill_value=0).reset_index()
count_df["csfbiased_pct"] = 100*count_df["csf_biased"] / (count_df["csf_biased"]+count_df["neutral"]+count_df["pb_biased"])
count_df

#### Subgoal 3: Plot % of clones that are CSF-biased per donor

In [ ]:
# Define manual colors
palette = {
    "NIC": "#FFFFFF",
    "MS": "#000000"
}

fig, ax = plt.subplots(figsize=(3, 4))

# Boxplot
sb.boxplot(
    data=count_df,
    x="catpat",
    y="csfbiased_pct",
    order=["NIC", "MS"],
    width=0.5,
    showcaps=True,
    showfliers=False,
    boxprops={"facecolor": "white", "edgecolor": "black"},
    whiskerprops={"color": "black"},
    capprops={"color": "black"},
    medianprops={"color": "black"},
    ax=ax
)

# Stripplot
sb.stripplot(
    data=count_df,
    x="catpat",
    y="csfbiased_pct",
    order=["NIC", "MS"],
    hue="catpat",
    hue_order=["NIC", "MS"],
    palette=palette,
    dodge=False,
    size=8,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("CSF-biased (%)")
ax.set_ylim(0, 100)
sb.despine()
ax.grid(False)

plt.tight_layout()
plt.show()

#### Over different expansion thresholds

In [ ]:
## Prune columns / drop duplicates
# df_input = df_input[["donor", "catpat", "clone_handle", "total_clone_count", "csf_comp_pct",  "csf_threshold", "pb_threshold", "biased_ann"]].drop_duplicates()

# # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# # Enumerate per donor how many CSF-biased clones each donor has over different expansion thresholds
# # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
count_df = df_input[["donor", "catpat", "biased_ann", "total_clone_count"]].value_counts().reset_index()
count_df = count_df.pivot_table(index = ["donor", "catpat", "total_clone_count"], columns = "biased_ann", values = "count", fill_value=0).reset_index()

## Make 5 dfs
df0 = count_df[count_df["total_clone_count"] > 0].groupby(["donor", "catpat"], observed=True)[["csf_biased", "neutral", "pb_biased"]].sum().reset_index()
df0["source"] = "0"

df5 = count_df[count_df["total_clone_count"] > 5].groupby(["donor", "catpat"], observed=True)[["csf_biased", "neutral", "pb_biased"]].sum().reset_index()
df5["source"] = "5"

df10 = count_df[count_df["total_clone_count"] > 10].groupby(["donor", "catpat"], observed=True)[["csf_biased", "neutral", "pb_biased"]].sum().reset_index()
df10["source"] = "10"

df15 = count_df[count_df["total_clone_count"] > 15].groupby(["donor", "catpat"], observed=True)[["csf_biased", "neutral", "pb_biased"]].sum().reset_index()
df15["source"] = "15"

df20 = count_df[count_df["total_clone_count"] > 20].groupby(["donor", "catpat"], observed=True)[["csf_biased", "neutral", "pb_biased"]].sum().reset_index()
df20["source"] = "20"

### sum_df
sum_df = pd.concat([df0, df5, df10, df15, df20])

### frequency of csf biased clones
sum_df["csfbiased_pct"] = 100*sum_df["csf_biased"] / (sum_df["csf_biased"]+sum_df["neutral"]+sum_df["pb_biased"])

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
#             plot
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Define manual colors
palette = {
    "NIC": "#FFFFFF",   # blue
    "MS": "#000000"       # red
}

fig, ax = plt.subplots(figsize=(5,4))

# Boxplot (keep white boxes)
sb.boxplot(
    data=sum_df,
    x="source",
    y="csfbiased_pct",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    # log_scale=True,
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (colored dots)
sb.stripplot(
    data=sum_df,
    x="source",
    y="csfbiased_pct",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    # log_scale=True,
    s=10,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette, 
    ax=ax
)

# Remove duplicate legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="Group")

ax.set_xlabel("Expansion thresholds (n cells > x)")
ax.set_ylabel("% CSF-biased clones") #% of cells in CSF\nnorm. to clone sizes

sb.despine(ax=ax)
ax.grid(False)
plt.tight_layout()

fig.savefig(
    "../../05_plots/13_final_plots/XX_boxplot_freq_of_CSF_biased_clones.pdf",
    bbox_inches='tight',
    format="pdf",
    dpi=400
)

plt.show()

#Save a list of expansion thresholds
threshold_order = ["0", "5", "10", "15", "20"]

# Initialize list to collect results
stats_list = []

for xtick in threshold_order:
    data_xtick = sum_df[sum_df["source"] == xtick]
    
    MS_vals = data_xtick[data_xtick["catpat"] == "MS"]["csfbiased_pct"]
    ctrl_vals = data_xtick[data_xtick["catpat"] == "NIC"]["csfbiased_pct"]
    
    if len(MS_vals) > 0 and len(ctrl_vals) > 0:
        stat, pval = mannwhitneyu(MS_vals, ctrl_vals, alternative='two-sided')
        
        stats_list.append({
            "expansion_threshold": xtick,
            "statistic": stat,
            "p_value": pval,
            "median_MS": MS_vals.median(),
            "median_NIC": ctrl_vals.median(),
            "n_MS": len(MS_vals),
            "n_NIC": len(ctrl_vals)
        })

# Convert to dataframe
df_stats = pd.DataFrame(stats_list)

# Apply Benjamini-Hochberg correction
reject, pvals_bh, _, _ = multipletests(
    df_stats["p_value"],
    alpha=0.05,
    method="fdr_bh"
)

# Add results to dataframe
df_stats["p_value_BH"] = pvals_bh
df_stats["significant_BH"] = reject

df_stats

#### Subgoal 4: Fixed threshold % CSF-biased out of all clones

In [ ]:
## Get obs_df
obs_df = sdata["gex"].obs
obs_df = obs_df[obs_df["catpat"] != "IC"]
obs_df = obs_df[obs_df["clone_handle"].notna()]

## Select columns
obs_df = obs_df[["donor", "catpat", "clone_handle", "Compartment", "type_x_clone_count", "total_clone_count"]]

# Remove all unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = obs_df[col].cat.remove_unused_categories()

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Compute Compartment Percentages
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Compute % contribution of each compartment per clone
obs_df["Compartment_pct"] = 100 * obs_df["type_x_clone_count"] / obs_df["total_clone_count"]

# Create input dataframe for pivoting
df_input = obs_df[["donor", "clone_handle", "total_clone_count", "Compartment", "Compartment_pct", "catpat"]].drop_duplicates()

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Add static statistical thresholds
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
df_input["biased_ann"] = np.where((df_input["Compartment"]== "CSF") & (df_input["Compartment_pct"] >= 53.8), "csf_biased", 
                                  np.where((df_input["Compartment"]== "PB") & (df_input["Compartment_pct"] == 100), "pb_biased", "neutral"))

# # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# # Enumerate per donor how many CSF-biased clones each donor has
# # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
count_df = df_input[["donor", "catpat", "biased_ann"]].value_counts().reset_index()
count_df = count_df.pivot_table(index = ["donor", "catpat"], columns = "biased_ann", values = "count", fill_value=0).reset_index()
count_df["csfbiased_pct"] = 100*count_df["csf_biased"] / (count_df["csf_biased"]+count_df["neutral"]+count_df["pb_biased"])
count_df

#### Over different expansion thresholds

In [ ]:
## Prune columns / drop duplicates
# df_input = df_input[["donor", "catpat", "clone_handle", "total_clone_count", "csf_comp_pct",  "csf_threshold", "pb_threshold", "biased_ann"]].drop_duplicates()

# # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# # Enumerate per donor how many CSF-biased clones each donor has over different expansion thresholds
# # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
count_df = df_input[["donor", "catpat", "biased_ann", "total_clone_count"]].value_counts().reset_index()
count_df = count_df.pivot_table(index = ["donor", "catpat", "total_clone_count"], columns = "biased_ann", values = "count", fill_value=0).reset_index()

## Make 5 dfs
df0 = count_df[count_df["total_clone_count"] > 0].groupby(["donor", "catpat"], observed=True)[["csf_biased", "neutral", "pb_biased"]].sum().reset_index()
df0["source"] = "0"

df5 = count_df[count_df["total_clone_count"] > 5].groupby(["donor", "catpat"], observed=True)[["csf_biased", "neutral", "pb_biased"]].sum().reset_index()
df5["source"] = "5"

df10 = count_df[count_df["total_clone_count"] > 10].groupby(["donor", "catpat"], observed=True)[["csf_biased", "neutral", "pb_biased"]].sum().reset_index()
df10["source"] = "10"

df15 = count_df[count_df["total_clone_count"] > 15].groupby(["donor", "catpat"], observed=True)[["csf_biased", "neutral", "pb_biased"]].sum().reset_index()
df15["source"] = "15"

df20 = count_df[count_df["total_clone_count"] > 20].groupby(["donor", "catpat"], observed=True)[["csf_biased", "neutral", "pb_biased"]].sum().reset_index()
df20["source"] = "20"

### sum_df
sum_df = pd.concat([df0, df5, df10, df15, df20])

### frequency of csf biased clones
sum_df["csfbiased_pct"] = 100*sum_df["csf_biased"] / (sum_df["csf_biased"]+sum_df["neutral"]+sum_df["pb_biased"])

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
#             plot
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Define manual colors
palette = {
    "NIC": "#FFFFFF",   # blue
    "MS": "#000000"       # red
}

fig, ax = plt.subplots(figsize=(5,4))

# Boxplot (keep white boxes)
sb.boxplot(
    data=sum_df,
    x="source",
    y="csfbiased_pct",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    # log_scale=True,
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (colored dots)
sb.stripplot(
    data=sum_df,
    x="source",
    y="csfbiased_pct",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    # log_scale=True,
    s=10,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette, 
    ax=ax
)

# Remove duplicate legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="Group")

ax.set_xlabel("Expansion thresholds (n cells > x)")
ax.set_ylabel("% CSF-biased clones") #% of cells in CSF\nnorm. to clone sizes

sb.despine(ax=ax)
ax.grid(False)
plt.tight_layout()

fig.savefig(
    "../../05_plots/13_final_plots/XX_boxplot_freq_of_CSF_biased_clones.pdf",
    bbox_inches='tight',
    format="pdf",
    dpi=400
)

plt.show()

#Save a list of expansion thresholds
threshold_order = ["0", "5", "10", "15", "20"]

# Initialize list to collect results
stats_list = []

for xtick in threshold_order:
    data_xtick = sum_df[sum_df["source"] == xtick]
    
    MS_vals = data_xtick[data_xtick["catpat"] == "MS"]["csfbiased_pct"]
    ctrl_vals = data_xtick[data_xtick["catpat"] == "NIC"]["csfbiased_pct"]
    
    if len(MS_vals) > 0 and len(ctrl_vals) > 0:
        stat, pval = mannwhitneyu(MS_vals, ctrl_vals, alternative='two-sided')
        
        stats_list.append({
            "expansion_threshold": xtick,
            "statistic": stat,
            "p_value": pval,
            "median_MS": MS_vals.median(),
            "median_NIC": ctrl_vals.median(),
            "n_MS": len(MS_vals),
            "n_NIC": len(ctrl_vals)
        })

# Convert to dataframe
df_stats = pd.DataFrame(stats_list)

# Apply Benjamini-Hochberg correction
reject, pvals_bh, _, _ = multipletests(
    df_stats["p_value"],
    alpha=0.05,
    method="fdr_bh"
)

# Add results to dataframe
df_stats["p_value_BH"] = pvals_bh
df_stats["significant_BH"] = reject

df_stats

# Plot 9: Number of cells in CSF per ml over expansion thresholds

#### Cells in CSF per ml over expansion thresholds

In [ ]:
## Get obs_df
obs_df = sdata["gex"].obs
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Select columns
obs_df = obs_df[["donor", "catpat", "clone_handle", "Compartment", "type_x_clone_count"]].drop_duplicates()

# Remove all unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = obs_df[col].cat.remove_unused_categories()

## Pivot
df_input = obs_df.pivot_table(index = ["donor", "catpat", "clone_handle"], columns="Compartment", values="type_x_clone_count", fill_value=0).reset_index()
df_input["total"] = df_input["CSF"] + df_input["PB"]

## Make 5 dataframes at different expansion thresholds
df0 =  df_input[df_input["total"] > 0].groupby(["donor", "catpat"], observed=True)["CSF"].sum().reset_index(name="cell_count")
df0["source"] = "0"

df5 =  df_input[df_input["total"] > 5].groupby(["donor", "catpat"], observed=True)["CSF"].sum().reset_index(name="cell_count")
df5["source"] = "5"

df10 =  df_input[df_input["total"] > 10].groupby(["donor", "catpat"], observed=True)["CSF"].sum().reset_index(name="cell_count")
df10["source"] = "10"

df15 =  df_input[df_input["total"] > 15].groupby(["donor", "catpat"], observed=True)["CSF"].sum().reset_index(name="cell_count")
df15["source"] = "15"

df20 =  df_input[df_input["total"] > 20].groupby(["donor", "catpat"], observed=True)["CSF"].sum().reset_index(name="cell_count")
df20["source"] = "20"

### sum_df
sum_df = pd.concat([df0, df5, df10, df15, df20])
sum_df.head(5)

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
sum_df["csf_vol"] = sum_df["donor"].map(csf_ml)

## normalize cell_count
sum_df["cell_count_norm"] = sum_df["cell_count"] / sum_df["csf_vol"]

In [ ]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
#             plot
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Define manual colors
palette = {
    "NIC": "#FFFFFF",   # blue
    "MS": "#000000"       # red
}

fig, ax = plt.subplots(figsize=(6.5,5))

# Boxplot (keep white boxes)
sb.boxplot(
    data=sum_df,
    x="source",
    y="cell_count_norm",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    log_scale=True,
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (colored dots)
sb.stripplot(
    data=sum_df,
    x="source",
    y="cell_count_norm",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    log_scale=True,
    s=10,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette, 
    ax=ax
)

# Remove duplicate legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="Group")

ax.set_xlabel("Expansion thresholds (n cells > x)")
ax.set_ylabel("CD8 T-cells in CSF/ml") #% of cells in CSF\nnorm. to clone sizes

sb.despine(ax=ax)
plt.tight_layout()

fig.savefig(
    "../../05_plots/13_final_plots/XX_boxplot_counts_of_CSF_biased_clones.pdf",
    bbox_inches='tight',
    format="pdf",
    dpi=400
)

plt.show()

#Save a list of expansion thresholds
threshold_order = ["0", "5", "10", "15", "20"]

# Initialize list to collect results
stats_list = []

for xtick in threshold_order:
    data_xtick = sum_df[sum_df["source"] == xtick]
    
    MS_vals = data_xtick[data_xtick["catpat"] == "MS"]["cell_count"]
    ctrl_vals = data_xtick[data_xtick["catpat"] == "NIC"]["cell_count"]
    
    if len(MS_vals) > 0 and len(ctrl_vals) > 0:
        stat, pval = mannwhitneyu(MS_vals, ctrl_vals, alternative='two-sided')
        
        stats_list.append({
            "expansion_threshold": xtick,
            "statistic": stat,
            "p_value": pval,
            "median_MS": MS_vals.median(),
            "median_NIC": ctrl_vals.median(),
            "n_MS": len(MS_vals),
            "n_NIC": len(ctrl_vals)
        })

# Convert to dataframe
df_stats = pd.DataFrame(stats_list)

# Apply Benjamini-Hochberg correction
reject, pvals_bh, _, _ = multipletests(
    df_stats["p_value"],
    alpha=0.05,
    method="fdr_bh"
)

# Add results to dataframe
df_stats["p_value_BH"] = pvals_bh
df_stats["significant_BH"] = reject

df_stats